# 04 – Interaktives Dashboard

Erstellt ein interaktives Plotly-Dashboard und exportiert es als
`output/polar_dashboard.html`.

## Inhalt
1. Setup & Daten laden
2. KPI-Karten
3. Chart 1 – Schritte (Monatsdurchschnitt)
4. Chart 2 – Ruhepuls-Trend
5. Chart 3 – Trainings pro Jahr
6. Chart 4 – Sportarten-Donut
7. Chart 5 – Heatmap Ruhepuls
8. Chart 6 – Schlafdauer-Trend
9. Dashboard zusammenbauen & exportieren

---
> 🎨 Dunkles Design. Alle Plotly-Daten als Python-Listen für korrekten HTML-Export.

## 1 – Setup & Daten laden

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))
from db_loader import DatabricksLoader, secrets_pruefen

secrets_pruefen()

# ── Farben (dunkles Design) ──────────────────────────────────
C = {
    'bg'       : '#0f1117',   # Hintergrund
    'card'     : '#1a1d27',   # Karten-Hintergrund
    'border'   : '#2d3142',   # Rahmen
    'text'     : '#e8eaf0',   # Haupttext
    'subtext'  : '#8892a4',   # Nebentext
    'blau'     : '#4f8ef7',   # Primärfarbe
    'blau_hell': '#7eb3ff',
    'orange'   : '#ff8c42',   # Akzent 1
    'gruen'    : '#43c59e',   # Akzent 2
    'rot'      : '#f25f5c',   # Akzent 3
    'lila'     : '#9d7fe3',   # Akzent 4
    'gelb'     : '#ffd166',   # Akzent 5
    'grid'     : '#1e2236',   # Gitter
}

# Gemeinsames Plotly-Layout-Template
LAYOUT_BASE = dict(
    paper_bgcolor = C['bg'],
    plot_bgcolor  = C['card'],
    font          = dict(family='Inter, Arial, sans-serif',
                         color=C['text'], size=12),
    title_font    = dict(size=15, color=C['text']),
    margin        = dict(l=50, r=30, t=50, b=40),
    xaxis         = dict(gridcolor=C['grid'], linecolor=C['border'],
                         tickcolor=C['border'], tickfont=dict(color=C['subtext'])),
    yaxis         = dict(gridcolor=C['grid'], linecolor=C['border'],
                         tickcolor=C['border'], tickfont=dict(color=C['subtext'])),
    legend        = dict(bgcolor='rgba(0,0,0,0)', bordercolor=C['border'],
                         font=dict(color=C['subtext'])),
    hoverlabel    = dict(bgcolor=C['card'], font_color=C['text'],
                         bordercolor=C['border']),
)

# ── Daten laden ──────────────────────────────────────────────
db = DatabricksLoader()

df_act   = db.lade_activity()
df_train = db.lade_training()
df_hr    = db.lade_heartrate()
df_hrv   = db.lade_hrv()
df_monat = db.monatsaggregat_activity()

for df in [df_act, df_train, df_hr, df_hrv]:
    if not df.empty:
        df['datum'] = pd.to_datetime(df['datum'])

print("✅ Daten geladen – starte Dashboard-Erstellung")

## 2 – KPI-Werte berechnen

In [ ]:
# ── KPI-Werte als einfache Python-Variablen ──────────────────
# (Kein pandas Series im HTML-Export → alles in native Python konvertieren)

kpi = {}

if not df_act.empty:
    kpi['jahre']       = round((df_act['datum'].max() - df_act['datum'].min()).days / 365.25, 1)
    kpi['schritte_avg']= int(round(float(df_act['schritte'].mean()), 0))
    kpi['schlaf_avg']  = round(float(df_act['schlaf_stunden'].mean()), 1)
else:
    kpi['jahre'] = kpi['schritte_avg'] = kpi['schlaf_avg'] = 0

if not df_hr.empty:
    kpi['ruhepuls_avg'] = round(float(df_hr['hr_ruhepuls'].mean()), 1)
else:
    kpi['ruhepuls_avg'] = 0

if not df_train.empty:
    kpi['trainings_n'] = int(len(df_train))
    kpi['laeufe_n']    = int(len(df_train[df_train['sport'] == 'RUNNING']))
else:
    kpi['trainings_n'] = kpi['laeufe_n'] = 0

print("KPI-Übersicht:")
for k, v in kpi.items():
    print(f"  {k:<18}: {v}")

## 3 – Chart 1: Schritte (Monatsdurchschnitt)

In [ ]:
fig_schritte = go.Figure()

if not df_monat.empty:
    # Datum aus Jahr + Monat zusammensetzen → als Python-Listen
    datum_liste    = [f"{int(r.jahr):04d}-{int(r.monat):02d}-01"
                      for r in df_monat.itertuples()]
    schritte_liste = [float(v) if not pd.isna(v) else None
                      for v in df_monat['schritte_avg']]

    # Balken
    fig_schritte.add_trace(go.Bar(
        x=datum_liste,
        y=schritte_liste,
        name='Ø Schritte',
        marker_color=C['blau'],
        marker_opacity=0.75,
        hovertemplate='%{x|%b %Y}<br>%{y:,.0f} Schritte<extra></extra>',
    ))

    # 6-Monats-Glättung als Linie
    schritte_series = pd.Series(schritte_liste)
    trend_liste = [float(v) if not pd.isna(v) else None
                   for v in schritte_series.rolling(6, min_periods=2, center=True).mean()]
    fig_schritte.add_trace(go.Scatter(
        x=datum_liste,
        y=trend_liste,
        name='6-Monats-Trend',
        mode='lines',
        line=dict(color=C['orange'], width=2.5),
        hovertemplate='%{x|%b %Y}<br>Trend: %{y:,.0f}<extra></extra>',
    ))

    # 10'000-Ziellinie
    fig_schritte.add_hline(
        y=10000,
        line=dict(color=C['gruen'], width=1.5, dash='dash'),
        annotation_text="10'000 Ziel",
        annotation_font=dict(color=C['gruen'], size=11),
        annotation_position='top right',
    )

fig_schritte.update_layout(
    **LAYOUT_BASE,
    title='📶 Schritte – Monatsdurchschnitt',
    xaxis_title='Monat',
    yaxis_title='Ø Schritte / Tag',
    yaxis_tickformat=',',
    barmode='overlay',
    height=380,
)

fig_schritte.show()
print("✅ Chart 1 erstellt")

## 4 – Chart 2: Ruhepuls-Trend

In [ ]:
fig_hr = go.Figure()

if not df_hr.empty:
    df_hr_s = df_hr.sort_values('datum').copy()

    # Rohwerte als Python-Listen (Scatter, dünn)
    datum_hr   = [str(d.date()) for d in df_hr_s['datum']]
    ruhepuls   = [float(v) if not pd.isna(v) else None
                  for v in df_hr_s['hr_ruhepuls']]

    fig_hr.add_trace(go.Scatter(
        x=datum_hr, y=ruhepuls,
        mode='markers',
        name='Tageswert',
        marker=dict(color=C['blau'], size=2.5, opacity=0.25),
        hovertemplate='%{x}<br>%{y:.0f} bpm<extra></extra>',
    ))

    # 6-Monats-Glättung
    trend_6m = [float(v) if not pd.isna(v) else None
                for v in pd.Series(ruhepuls)
                .rolling(180, min_periods=30, center=True).mean()]
    fig_hr.add_trace(go.Scatter(
        x=datum_hr, y=trend_6m,
        mode='lines',
        name='6-Monats-Glättung',
        line=dict(color=C['orange'], width=2.5),
        hovertemplate='%{x}<br>Trend: %{y:.1f} bpm<extra></extra>',
    ))

    # Konfidenzband (IQR)
    p25 = [float(v) if not pd.isna(v) else None
           for v in pd.Series(ruhepuls)
           .rolling(180, min_periods=30, center=True).quantile(0.25)]
    p75 = [float(v) if not pd.isna(v) else None
           for v in pd.Series(ruhepuls)
           .rolling(180, min_periods=30, center=True).quantile(0.75)]

    fig_hr.add_trace(go.Scatter(
        x=datum_hr + datum_hr[::-1],
        y=p75 + p25[::-1],
        fill='toself',
        fillcolor=f'rgba(79,142,247,0.10)',
        line=dict(color='rgba(0,0,0,0)'),
        name='IQR (25–75%)',
        hoverinfo='skip',
    ))

fig_hr.update_layout(
    **LAYOUT_BASE,
    title='❤️ Ruhepuls-Entwicklung',
    xaxis_title='Datum',
    yaxis_title='Ruhepuls (bpm)',
    height=380,
    legend=dict(orientation='h', y=1.05, x=0,
                bgcolor='rgba(0,0,0,0)',
                font=dict(color=C['subtext'])),
)

fig_hr.show()
print("✅ Chart 2 erstellt")

## 5 – Chart 3: Trainings pro Jahr

In [ ]:
fig_training = go.Figure()

if not df_train.empty:
    df_train['jahr'] = df_train['datum'].dt.year

    # Top-5 Sportarten
    top5 = df_train['sport'].value_counts().head(5).index.tolist()
    farben_sports = [C['blau'], C['orange'], C['gruen'], C['lila'], C['gelb']]

    for sport, farbe in zip(top5, farben_sports):
        df_s   = df_train[df_train['sport'] == sport].groupby('jahr').size()
        jahre  = [int(j) for j in df_s.index.tolist()]
        anzahl = [int(n) for n in df_s.values.tolist()]

        fig_training.add_trace(go.Bar(
            x=jahre, y=anzahl,
            name=sport.title(),
            marker_color=farbe,
            marker_opacity=0.85,
            hovertemplate=f'{sport.title()}<br>%{{x}}: %{{y}} Einheiten<extra></extra>',
        ))

    # Übrige Sportarten summiert
    df_rest = df_train[~df_train['sport'].isin(top5)].groupby('jahr').size()
    if not df_rest.empty:
        fig_training.add_trace(go.Bar(
            x=[int(j) for j in df_rest.index.tolist()],
            y=[int(n) for n in df_rest.values.tolist()],
            name='Andere',
            marker_color=C['subtext'],
            marker_opacity=0.6,
            hovertemplate='Andere<br>%{x}: %{y} Einheiten<extra></extra>',
        ))

fig_training.update_layout(
    **LAYOUT_BASE,
    title='🏃 Trainingseinheiten pro Jahr',
    xaxis_title='Jahr',
    yaxis_title='Anzahl Einheiten',
    barmode='stack',
    height=380,
    xaxis=dict(
        **LAYOUT_BASE['xaxis'],
        tickmode='linear', dtick=1,
    ),
)

fig_training.show()
print("✅ Chart 3 erstellt")

## 6 – Chart 4: Sportarten-Donut

In [ ]:
fig_donut = go.Figure()

if not df_train.empty:
    sport_counts = df_train['sport'].value_counts()

    # Top-7 anzeigen, Rest zusammenfassen
    top7         = sport_counts.head(7)
    rest_n       = int(sport_counts.iloc[7:].sum()) if len(sport_counts) > 7 else 0

    labels = [str(s).title() for s in top7.index.tolist()]
    values = [int(v) for v in top7.values.tolist()]

    if rest_n > 0:
        labels.append('Andere')
        values.append(rest_n)

    donut_farben = [
        C['blau'], C['orange'], C['gruen'], C['lila'],
        C['gelb'], C['rot'], C['blau_hell'], C['subtext']
    ]

    fig_donut.add_trace(go.Pie(
        labels=labels,
        values=values,
        hole=0.55,
        marker=dict(
            colors=donut_farben[:len(labels)],
            line=dict(color=C['bg'], width=2)
        ),
        textinfo='label+percent',
        textfont=dict(color=C['text'], size=11),
        hovertemplate='%{label}<br>%{value} Einheiten (%{percent})<extra></extra>',
        insidetextorientation='auto',
    ))

    # Gesamtzahl in der Mitte
    fig_donut.add_annotation(
        text=f'<b>{sum(values)}</b><br>Trainings',
        x=0.5, y=0.5,
        font=dict(size=18, color=C['text']),
        showarrow=False,
    )

fig_donut.update_layout(
    **LAYOUT_BASE,
    title='🥇 Sportarten-Verteilung',
    height=400,
    showlegend=True,
    legend=dict(
        orientation='v', x=1.02, y=0.5,
        font=dict(color=C['subtext'], size=10),
        bgcolor='rgba(0,0,0,0)',
    ),
)

fig_donut.show()
print("✅ Chart 4 erstellt")

## 7 – Chart 5: Heatmap Ruhepuls nach Monat & Wochentag

In [ ]:
fig_heatmap = go.Figure()

if not df_hr.empty:
    wochentage_de = ['Mo', 'Di', 'Mi', 'Do', 'Fr', 'Sa', 'So']
    monate_de     = ['Jan','Feb','Mär','Apr','Mai','Jun',
                     'Jul','Aug','Sep','Okt','Nov','Dez']

    # Pivot berechnen
    pivot = (
        df_hr.groupby(['monat','wochentag_nr'])['hr_ruhepuls']
        .mean()
        .unstack()
    )

    # Sicherstellen dass alle Spalten/Zeilen vorhanden
    pivot = pivot.reindex(index=range(1,13), columns=range(0,7))

    # Als Python-Listen (kein pandas für JSON-Export)
    z_werte = [[round(float(v), 1) if not pd.isna(v) else None
                for v in row]
               for row in pivot.values.tolist()]

    fig_heatmap.add_trace(go.Heatmap(
        z=z_werte,
        x=wochentage_de,
        y=monate_de,
        colorscale=[
            [0.0, '#43c59e'],   # tief → grün
            [0.5, '#ffd166'],   # mittel → gelb
            [1.0, '#f25f5c'],   # hoch → rot
        ],
        colorbar=dict(
            title=dict(text='bpm', font=dict(color=C['subtext'])),
            tickfont=dict(color=C['subtext']),
            bgcolor=C['card'],
            bordercolor=C['border'],
        ),
        hovertemplate='%{y} %{x}<br>Ø Ruhepuls: %{z:.1f} bpm<extra></extra>',
        text=[[f"{v:.1f}" if v is not None else ''
               for v in row] for row in z_werte],
        texttemplate='%{text}',
        textfont=dict(size=10, color=C['bg']),
    ))

fig_heatmap.update_layout(
    **LAYOUT_BASE,
    title='🌡️ Ruhepuls-Heatmap: Monat × Wochentag',
    xaxis=dict(
        **LAYOUT_BASE['xaxis'],
        title='Wochentag',
        side='bottom',
    ),
    yaxis=dict(
        **LAYOUT_BASE['yaxis'],
        title='Monat',
        autorange='reversed',
    ),
    height=420,
)

fig_heatmap.show()
print("✅ Chart 5 erstellt")

## 8 – Chart 6: Schlafdauer-Trend

In [ ]:
fig_schlaf = go.Figure()

if not df_monat.empty:
    datum_liste  = [f"{int(r.jahr):04d}-{int(r.monat):02d}-01"
                    for r in df_monat.itertuples()]
    schlaf_liste = [float(v) if not pd.isna(v) else None
                    for v in df_monat['schlaf_avg']]
    schlafq_liste= [float(v) if not pd.isna(v) else None
                    for v in df_monat['schlaf_q_avg']]

    # Schlafdauer Linie
    fig_schlaf.add_trace(go.Scatter(
        x=datum_liste, y=schlaf_liste,
        mode='lines',
        name='Ø Schlafdauer',
        line=dict(color=C['lila'], width=2),
        fill='tozeroy',
        fillcolor='rgba(157,127,227,0.10)',
        hovertemplate='%{x|%b %Y}<br>%{y:.2f}h<extra></extra>',
        yaxis='y1',
    ))

    # 6-Monats-Trend
    schlaf_trend = [float(v) if not pd.isna(v) else None
                    for v in pd.Series(schlaf_liste)
                    .rolling(6, min_periods=2, center=True).mean()]
    fig_schlaf.add_trace(go.Scatter(
        x=datum_liste, y=schlaf_trend,
        mode='lines',
        name='6-Monats-Trend',
        line=dict(color=C['blau_hell'], width=2.5, dash='dot'),
        hovertemplate='%{x|%b %Y}<br>Trend: %{y:.2f}h<extra></extra>',
        yaxis='y1',
    ))

    # Schlafqualität als Balken (zweite Y-Achse)
    fig_schlaf.add_trace(go.Bar(
        x=datum_liste, y=schlafq_liste,
        name='Ø Schlafqualität',
        marker_color=C['gruen'],
        marker_opacity=0.35,
        hovertemplate='%{x|%b %Y}<br>Qualität: %{y:.2f}<extra></extra>',
        yaxis='y2',
    ))

    # 7.5h-Empfehlungslinie
    fig_schlaf.add_hline(
        y=7.5, yref='y1',
        line=dict(color=C['orange'], width=1.5, dash='dash'),
        annotation_text='7.5h Empfehlung',
        annotation_font=dict(color=C['orange'], size=11),
        annotation_position='top right',
    )

fig_schlaf.update_layout(
    **LAYOUT_BASE,
    title='😴 Schlafdauer & -qualität',
    height=380,
    yaxis  = dict(**LAYOUT_BASE['yaxis'],
                  title='Schlafdauer (h)', side='left'),
    yaxis2 = dict(title='Schlafqualität (0–1)',
                  overlaying='y', side='right', range=[0, 2],
                  showgrid=False,
                  tickfont=dict(color=C['subtext']),
                  titlefont=dict(color=C['subtext'])),
    legend=dict(orientation='h', y=1.05, x=0,
                bgcolor='rgba(0,0,0,0)',
                font=dict(color=C['subtext'])),
)

fig_schlaf.show()
print("✅ Chart 6 erstellt")

## 9 – Dashboard zusammenbauen & exportieren

In [ ]:
# ── KPI-HTML-Karten ──────────────────────────────────────────
def kpi_karte(titel: str, wert: str, einheit: str, farbe: str) -> str:
    """
    Erstellt eine HTML-KPI-Karte.

    Args:
        titel:   Beschriftung der Karte (z.B. 'Ø Ruhepuls')
        wert:    Angezeigter Zahlenwert als String
        einheit: Einheit (z.B. 'bpm', 'h', 'Schritte')
        farbe:   CSS-Akzentfarbe

    Returns:
        HTML-String der Karte.
    """
    return f"""
    <div style="
        background:{C['card']};
        border:1px solid {C['border']};
        border-top:3px solid {farbe};
        border-radius:8px;
        padding:20px 24px;
        text-align:center;
        min-width:130px;
        flex:1;
    ">
        <div style="color:{C['subtext']};font-size:11px;
                    letter-spacing:1px;text-transform:uppercase;
                    margin-bottom:8px;">{titel}</div>
        <div style="color:{farbe};font-size:32px;
                    font-weight:700;line-height:1.1;">{wert}</div>
        <div style="color:{C['subtext']};font-size:12px;
                    margin-top:4px;">{einheit}</div>
    </div>
    """

kpi_html = f"""
<div style="display:flex;gap:16px;flex-wrap:wrap;margin-bottom:8px;">
    {kpi_karte('Datenspanne',    str(kpi.get('jahre', '–')),            'Jahre',    C['blau'])}
    {kpi_karte('Ø Schritte',     f"{kpi.get('schritte_avg',0):,}",      '/ Tag',    C['gruen'])}
    {kpi_karte('Ø Ruhepuls',     str(kpi.get('ruhepuls_avg', '–')),     'bpm',      C['rot'])}
    {kpi_karte('Ø Schlaf',       str(kpi.get('schlaf_avg', '–')),       'Stunden',  C['lila'])}
    {kpi_karte('Trainings',      f"{kpi.get('trainings_n',0):,}",       'gesamt',   C['orange'])}
    {kpi_karte('Läufe',          f"{kpi.get('laeufe_n',0):,}",          'gesamt',   C['gelb'])}
</div>
"""

print("✅ KPI-Karten erstellt")

In [ ]:
# ── Plotly-Figures zu HTML konvertieren ──────────────────────
# full_html=False → nur das <div>, kein komplettes HTML-Dokument
# include_plotlyjs='cdn' → Plotly.js via CDN (kleine Dateigrösse)

def fig_zu_html(fig: go.Figure, include_js: bool = False) -> str:
    """Konvertiert eine Plotly-Figure zu einem HTML-Div-String."""
    return fig.to_html(
        full_html=False,
        include_plotlyjs='cdn' if include_js else False,
        config={
            'displayModeBar' : True,
            'modeBarButtonsToRemove': ['sendDataToCloud', 'lasso2d', 'select2d'],
            'displaylogo'    : False,
            'responsive'     : True,
        }
    )

# Erste Figure lädt Plotly.js (einmalig)
html_schritte = fig_zu_html(fig_schritte, include_js=True)
html_hr       = fig_zu_html(fig_hr)
html_training = fig_zu_html(fig_training)
html_donut    = fig_zu_html(fig_donut)
html_heatmap  = fig_zu_html(fig_heatmap)
html_schlaf   = fig_zu_html(fig_schlaf)

print("✅ Alle Charts zu HTML konvertiert")

In [ ]:
# ── Vollständiges HTML-Dashboard zusammenbauen ───────────────
erstellt_am = datetime.now().strftime('%d.%m.%Y %H:%M')

dashboard_html = f"""<!DOCTYPE html>
<html lang="de">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Polar Health Analytics – Dashboard</title>
    <style>
        *, *::before, *::after {{ box-sizing: border-box; }}

        body {{
            margin: 0;
            padding: 24px;
            background: {C['bg']};
            color: {C['text']};
            font-family: Inter, -apple-system, Arial, sans-serif;
            font-size: 14px;
            line-height: 1.5;
        }}

        .header {{
            margin-bottom: 28px;
            padding-bottom: 20px;
            border-bottom: 1px solid {C['border']};
        }}

        .header h1 {{
            margin: 0 0 6px 0;
            font-size: 26px;
            font-weight: 700;
            background: linear-gradient(135deg, {C['blau']}, {C['lila']});
            -webkit-background-clip: text;
            -webkit-text-fill-color: transparent;
            background-clip: text;
        }}

        .header .meta {{
            color: {C['subtext']};
            font-size: 13px;
        }}

        .section {{
            margin-bottom: 28px;
        }}

        .section-title {{
            font-size: 11px;
            font-weight: 600;
            letter-spacing: 1.5px;
            text-transform: uppercase;
            color: {C['subtext']};
            margin-bottom: 12px;
            padding-left: 4px;
            border-left: 3px solid {C['blau']};
            padding-left: 10px;
        }}

        .chart-card {{
            background: {C['card']};
            border: 1px solid {C['border']};
            border-radius: 10px;
            padding: 8px;
            margin-bottom: 20px;
        }}

        .grid-2 {{
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 20px;
        }}

        .footer {{
            margin-top: 40px;
            padding-top: 16px;
            border-top: 1px solid {C['border']};
            color: {C['subtext']};
            font-size: 12px;
            text-align: center;
        }}

        @media (max-width: 900px) {{
            .grid-2 {{ grid-template-columns: 1fr; }}
            body {{ padding: 12px; }}
        }}
    </style>
</head>
<body>

    <!-- Header -->
    <div class="header">
        <h1>⌚ Polar Health Analytics</h1>
        <div class="meta">
            Persönliche Gesundheitsdaten 2014–2026 &nbsp;·&nbsp;
            Erstellt am {erstellt_am} &nbsp;·&nbsp;
            HSLU Data Science
        </div>
    </div>

    <!-- KPIs -->
    <div class="section">
        <div class="section-title">Kennzahlen</div>
        {kpi_html}
    </div>

    <!-- Chart 1: Schritte -->
    <div class="section">
        <div class="section-title">Aktivität</div>
        <div class="chart-card">{html_schritte}</div>
    </div>

    <!-- Chart 2: Ruhepuls -->
    <div class="section">
        <div class="section-title">Herzfrequenz</div>
        <div class="chart-card">{html_hr}</div>
    </div>

    <!-- Charts 3 & 4: Training + Donut -->
    <div class="section">
        <div class="section-title">Training</div>
        <div class="grid-2">
            <div class="chart-card">{html_training}</div>
            <div class="chart-card">{html_donut}</div>
        </div>
    </div>

    <!-- Chart 5: Heatmap -->
    <div class="section">
        <div class="section-title">Muster</div>
        <div class="chart-card">{html_heatmap}</div>
    </div>

    <!-- Chart 6: Schlaf -->
    <div class="section">
        <div class="section-title">Schlaf</div>
        <div class="chart-card">{html_schlaf}</div>
    </div>

    <!-- Footer -->
    <div class="footer">
        Polar Health Analytics &nbsp;·&nbsp;
        Daten: Polar Flow Export &nbsp;·&nbsp;
        Stack: Python · Databricks Delta Lake · Plotly &nbsp;·&nbsp;
        <a href="https://github.com" style="color:{C['blau']};
               text-decoration:none;">GitHub</a>
    </div>

</body>
</html>
"""

# ── Datei speichern ──────────────────────────────────────────
output_pfad = Path('..') / 'output' / 'polar_dashboard.html'
output_pfad.parent.mkdir(exist_ok=True)

with open(output_pfad, 'w', encoding='utf-8') as f:
    f.write(dashboard_html)

groesse_kb = output_pfad.stat().st_size / 1024
print(f"\n✅ Dashboard exportiert: {output_pfad.resolve()}")
print(f"   Dateigrösse: {groesse_kb:.0f} KB")
print(f"   Erstellt am: {erstellt_am}")
print(f"\n→ Im Codespace: Rechtsklick auf die Datei → 'Open with Live Server'")
print(f"→ Oder: python -m http.server 8080 im Terminal, dann Port 8080 öffnen")

db.schliessen()
print("\n🎉 Dashboard fertig!")